
# 🚀 دفتر `main` — تجميع المشروع (البيانات ← النموذج ← التدريب ← التقييم)

هذا الدفتر هو نقطة الدخول الوحيدة لتشغيل المشروع كاملاً. لا يُعرِّف منطقاً
جديداً بنفسه — **يستورد** الدفاتر الأربعة الأخرى (`crypto_data_pipeline_v6`،
`model_v2 (1)`، `trainer_framework_v2`، `chicks_v4_5_input_output_patterns`)
عبر `%run`، ويربط مخرَج كل واحد بمدخل التالي.

## لماذا احتاج الأمر طبقة "ربط" صريحة (لا استيراد مباشر فحسب)

تحليل الدفاتر الأربعة كشف نقاط عدم تطابق حقيقية في العقد بينها — ليست
أخطاء برمجية، بل فروقاً في التصميم نشأت لأن كل دفتر طُوِّر بمعزل عن الآخر:

1. **أُطر زمنية**: خط الأنابيب يدعم عدة أطر (`tf_order`)، لكن النموذج
   (`build_nig_timenet_v2`) يقبل مدخلاً واحداً فقط `(seq_len, n_features)`
   لفريم واحد. الحل هنا: القسم ٣ يشتقّ فريماً واحداً فعلياً من البيانات
   المحمَّلة (`dataset['timeframes']`)، لا افتراضاً مُثبَّتاً بالكود.

2. **تسمية الأهداف**: بيانات خط الأنابيب تحمل مفاتيح `y` بصيغة
   `y_{هدف}_{class|reg}` (مثلاً `y_close_reg`)، بينما مخرجات النموذج
   بصيغة `y_{هدف}` (بلا `_reg`، والاسم نفسه هو متوسط NIG لا خام مُقيَّس).
   `trainer_framework` يحسم هذا فعلاً بتصميمه: `true_key` (مفتاح البيانات)
   و`output_keys` (مفاتيح مخرجات النموذج) مساحتا تسمية منفصلتان تماماً —
   القسم ٥ هنا يربطهما صراحةً بدل افتراض تطابقهما.

3. **مقياس الهدف**: `reg_target_mode` الافتراضي في خط الأنابيب هو
   `'return'` — أي أن `y_close_reg` عائد نسبي مباشر (`(مستقبلي/آخر_سعر) - 1`)
   لا قيمة مُقيَّسة بـ`base_params` (تلك دلالة `'window_scale'` القديمة
   فقط). لكن دالة فكّ التشفير في `chicks` (`decode_predictions_v4`) مبنية
   على افتراض `pred_real = raw*iqr + median` — صيغة `base_params` القديمة.
   القسم ٦ هنا يمرّر `base_params = [آخر_سعر, آخر_سعر]` بدل
   `dataset['base_params']` الخام، فتُصبح نفس الصيغة تكافئ رياضياً عكس
   العائد المباشر (`آخر_سعر × (١ + عائد)`) — مطابقة تماماً لما تفعله
   `invert_reg_predictions` في خط الأنابيب لنفس الوضع.
   **بلا هذا التصحيح، كل تقرير من `chicks` كان سيُفسِّر عائداً صغيراً
   (~0.01) على أنه سعر مطلق — خطأ صامت لا يظهر كاستثناء.**

كل هذه القرارات مُختبَرة فعلياً (لا نظرياً فقط) في القسم الأخير من هذا
الدفتر ببيانات تركيبية بنفس الشكل — شغّله إن عدَّلت أي جزء من الربط أعلاه.


## ١) تحميل وتشغيل خط الأنابيب

In [ ]:
import os

REPO_URL = "https://github.com/yuosef772424/crypto-signal-prediction.git"
REPO_DIR = "/content/crypto-signal-prediction"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

In [ ]:
# يُعرِّف: CONFIG، build_dataset*، split_data، rolling_splits، mount_drive،
# save_data_to_drive/load_data_from_drive، invert_reg_predictions، ...
# ويُشغِّل تلقائياً run_pipeline_selftests() (بيانات تركيبية، بلا شبكة/Drive
# — آمن دائماً، ثوانٍ معدودة).
%run "crypto_data_pipeline_v6.ipynb"


## ٢) إعدادات هذا المشروع

عدّل هنا فوق افتراضيات خط الأنابيب — لا تُعدِّل `DEFAULT_CONFIG` مباشرة.
تحديد `tf_order`/`model_tf` مؤجَّل عمداً للقسم التالي (بعد تحميل البيانات
الفعلية)، لأن الفريم الصحيح يعتمد على ما بُني فعلاً في `dataset`، لا على
تخمين هنا.

⚠️ **ملاحظة مهمة**: تعديل `enabled_heads` هنا **لن** يُغيِّر بيانات
مُحمَّلة مسبقاً من Drive — ذلك الإعداد يؤثّر فقط على `build_dataset` وقت
بنائه الفعلي (في دفتر خط الأنابيب نفسه، قبل الحفظ). إن كنت تريد إيقاف
رؤوس تصنيف بعينها من الأساس، عدّله هناك وأعد بناء/حفظ البيانات — القسم ٥
هنا أصلاً يستخدم فقط مفاتيح `_reg` من `train['y']` ويتجاهل أي مفتاح
`_class` موجود، فلا يحتاج الأمر تعديلاً هنا لينجح.


In [ ]:
update_config({
    "project_name": "crypto_model",
    # أضف أي إعداد آخر يخصّ مشروعك هنا (مثلاً excluded_coins، eval_batch_size).
})
print("project_name:", CONFIG["project_name"])

## ٣) تحميل البيانات الجاهزة (من Drive) وتحديد الفريم الزمني الفعلي

In [ ]:
dataset = load_data_from_drive()  # يقرأ preprocessing_output_latest.pkl.gz

# النموذج الحالي (دفتر model_v2) يقبل فريماً واحداً فقط. نختار الفريم
# فعلياً من dataset['timeframes'] بدل تثبيته هنا: model_tf إن وُجد فعلاً
# في البيانات المحمَّلة، وإلا base_tf المُستخدَم فعلاً وقت البناء.
MODEL_TF = CONFIG.get("model_tf") if CONFIG.get("model_tf") in dataset["timeframes"] else dataset["base_tf"]
update_config({"tf_order": [MODEL_TF], "base_tf": MODEL_TF})

print(f"الأطر المتوفّرة في البيانات: {dataset['timeframes']} — الفريم المُستخدَم فعلياً: {MODEL_TF}")
print(f"عدد الميزات: {len(dataset['feature_order'])} — طول النافذة: {dataset['window_sizes'][MODEL_TF]}")

In [ ]:
train, val, test = split_data(dataset, config=CONFIG)
print("train:", train[f"X_{MODEL_TF}"].shape, "| val:", val[f"X_{MODEL_TF}"].shape,
      "| test assets:", list(test.keys()) if isinstance(test, dict) and all(isinstance(v, dict) and "y" in v for v in test.values()) else test[f"X_{MODEL_TF}"].shape)

## ٤) النموذج — `build_nig_timenet_v2` من دفتر `model_v2`

In [ ]:
# يُعرِّف: HEAD_REGISTRY، MODEL_CONFIG، build_model_fn، build_nig_timenet_v2، ...
# ويُشغِّل تلقائياً run_model_selftests() (بلا بيانات حقيقية — آمن دائماً).
%run "model_v2 (1).ipynb"

In [ ]:
SEQ_LEN = dataset["window_sizes"][MODEL_TF]
N_FEATURES = len(dataset["feature_order"])
PRICE_TARGETS = list(CONFIG["targets"])  # افتراضياً ['high', 'low', 'close']


def model_builder():
    """صفر-وسيط، كما يتطلّب build_training_system — نفس المعمارية كل مرّة
    (ضروري لصحّة استئناف الحالة المحفوظة)."""
    return build_model_fn(SEQ_LEN, N_FEATURES)


model = model_builder()
model.summary()


## ٥) التدريب — ربط مخرجات النموذج بمفاتيح بيانات خط الأنابيب

`true_key`: مفتاح الهدف في `train['y']` (من خط الأنابيب، بصيغة
`y_{هدف}_reg`). `output_keys`: أسماء مخرجات النموذج الفعلية (من دفتر
`model_v2`، بصيغة `y_{هدف}`/`_nu`/`_alpha`/`_beta`/`_confidence`).
مساحتا تسمية منفصلتان تماماً في تصميم `trainer_framework` — هذه الدالة
هي مكان الربط الوحيد، فتعديل تسمية أي طرف مستقبلاً يحتاج تعديلاً هنا فقط.


In [ ]:
def build_target_configs(price_targets):
    cfg = {}
    for t in price_targets:
        cfg[t] = {
            "true_key": f"y_{t}_reg",
            "task_type": "evidential",
            "output_keys": {
                "mu": f"y_{t}", "nu": f"y_{t}_nu", "alpha": f"y_{t}_alpha",
                "beta": f"y_{t}_beta", "confidence": f"y_{t}_confidence",
            },
        }
    return cfg

In [ ]:
# يُعرِّف: build_config، build_training_system، GenericTrainer، ...
# ويُشغِّل تلقائياً Smoke Test (بيانات تركيبية — عدّة ثوانٍ، آمن دائماً).
%run "trainer_framework_v2.ipynb"

In [ ]:
main_config = build_config({
    "run": {
        "run_dir": "/content/drive/MyDrive/training_runs/crypto_model_v1",  # ⚠️ غيّره لمسارك
        "epochs": 60,
        "batch_size": 64,
        "train_mode": "auto",
    },
    "targets": build_target_configs(PRICE_TARGETS),
})

import tensorflow as tf

def _y_for(split):
    return {cfg["true_key"]: split["y"][cfg["true_key"]] for cfg in main_config["targets"].values()}


train_ds = (tf.data.Dataset.from_tensor_slices((train[f"X_{MODEL_TF}"], _y_for(train)))
            .shuffle(4096).batch(main_config["run"]["batch_size"], drop_remainder=True)
            .prefetch(tf.data.AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((val[f"X_{MODEL_TF}"], _y_for(val)))
          .batch(main_config["run"]["batch_size"], drop_remainder=True)
          .prefetch(tf.data.AUTOTUNE))
sample_batch = next(iter(train_ds))

trainer, callbacks, initial_epoch = build_training_system(model_builder, main_config, sample_batch)
history = trainer.fit(
    train_ds, validation_data=val_ds, initial_epoch=initial_epoch,
    epochs=main_config["run"]["epochs"], callbacks=callbacks, verbose=1,
)
model = trainer.model


## ٦) الاختبار — تحويل تقسيم خط الأنابيب إلى شكل `chicks`

فجوتان يسدّهما المُحوِّل التالي (انظر شرح القسم ٠):
* تسمية `y`: `y_close_reg` (خط الأنابيب) ← `close` (اسم `TargetSpec` المجرَّد
  الذي يتوقّعه `chicks`).
* `base_params`: `[آخر_سعر, آخر_سعر]` بدل عمود `base_params` الخام في
  `dataset` — يجعل صيغة فكّ تشفير `chicks` (`raw*iqr + median`) تُكافئ
  عكس العائد المباشر (`آخر_سعر × (١ + عائد)`)، المطابق لـ`reg_target_mode='return'`.
  **صالح فقط لهذا الوضع** — إن حوّلت `reg_target_mode` إلى `'window_scale'`
  مرّر `split['base_params']` الخام بدل هذا التحويل.


In [ ]:
LAST_CLOSE_COL = LAST_COLUMNS.index("last_close")


def build_chicks_test_dict(pipeline_test, model_tf, reg_target_mode=None):
    """يحوّل `test` (مخرَج split_data، قاموس {أصل: قسم}) إلى الشكل الذي
    تتوقعه دوال chicks (`test_all_assets_v4`/`run_full_analysis`)."""
    reg_target_mode = reg_target_mode or CONFIG.get("reg_target_mode", "return")
    out = {}
    for asset, split in pipeline_test.items():
        last_candles = split["last_candles"]
        if reg_target_mode == "return":
            last_close = last_candles[:, LAST_CLOSE_COL]
            base_params_eval = np.stack([last_close, last_close], axis=1).astype("float32")
        else:
            base_params_eval = split["base_params"]
        out[asset] = {
            f"X_{model_tf}": split[f"X_{model_tf}"],
            "base_params": base_params_eval,
            "last_candles": last_candles,
            "y": {t: split["y"][f"y_{t}_reg"] for t in PRICE_TARGETS},
        }
    return out


import numpy as np

test_dict = build_chicks_test_dict(test, MODEL_TF)

In [ ]:
# يُعرِّف: TargetSpec، DEFAULT_PRICE_TARGETS، predict_with_evaluation_v4،
# test_all_assets_v4، predict_latest_v4/predict_latest_all_assets،
# run_full_analysis، ...
%run "chicks_v4_5_input_output_patterns.ipynb"

In [ ]:
full_results = run_full_analysis(
    model=model,
    test_dict=test_dict,
    timeframes=[MODEL_TF],
    target_specs=DEFAULT_PRICE_TARGETS,
    out_dir="analysis_outputs",
)
full_results["per_asset_results"]

## ٧) دوال فحص وتحقّق إضافية (تُستدعى عند الحاجة، لا تلقائياً)

In [ ]:
def latest_trading_report(n_display=5):
    """تقرير 'آخر N عيّنات' لكل أصل — للتداول الحيّ، يعمل بلا أهداف حقيقية."""
    return predict_latest_all_assets(
        model, test_dict, timeframes=[MODEL_TF], target_specs=DEFAULT_PRICE_TARGETS,
        n_display=n_display,
    )


def real_price_predictions(asset, target):
    """يحوّل مخرَج النموذج (عائد مباشر) لسعر حقيقي عبر invert_reg_predictions
    نفسها المستخدَمة في خط الأنابيب — مفيد حين تحتاج سعراً لا عائداً."""
    split = test[asset]
    x = split[f"X_{MODEL_TF}"]
    preds = model(x, training=False)[f"y_{target}"].numpy().ravel()
    return invert_reg_predictions(preds, target, last_candles=split["last_candles"], config=CONFIG)


## ٨) اختبار ذاتي للتوصيل بين الدفاتر (بيانات تركيبية — بلا Drive ولا تدريب حقيقي)

يبني بيانات بنفس شكل مخرَج خط الأنابيب تماماً، يُدرِّب نموذجاً حقيقياً
حقبتين، ثم يُغذّي `chicks` عبر نفس `build_chicks_test_dict` أعلاه —
يُثبت أن كل نقاط الربط الثلاث (القسم ٠) تعمل معاً فعلياً، لا افتراضاً.
شغّله بعد أي تعديل على قسم الربط في هذا الدفتر، أو على أي من الدفاتر الأربعة.


In [ ]:
def run_wiring_selftest(verbose=True):
    rng = np.random.default_rng(0)
    seq_len, n_features = 16, 10
    tf_name = "1D"

    def make_split(n):
        X = rng.normal(size=(n, seq_len, n_features)).astype("float32")
        last_close = 100 + rng.normal(size=(n,)) * 5
        last_high = last_close + np.abs(rng.normal(size=(n,)))
        last_low = last_close - np.abs(rng.normal(size=(n,)))
        ret = {t: rng.normal(scale=0.01, size=(n,)).astype("float32") for t in ("high", "low", "close")}
        last_candles = np.stack([
            last_high, last_low, last_close, np.arange(n).astype("float64"),
            last_close * (1 + ret["close"]), last_low * (1 + ret["low"]), last_high * (1 + ret["high"]),
        ], axis=1)
        return {
            f"X_{tf_name}": X,
            "y": {f"y_{t}_reg": v for t, v in ret.items()},
            "base_params": np.zeros((n, 2), dtype="float32"),
            "last_candles": last_candles,
        }

    fake_train, fake_val = make_split(256), make_split(64)
    fake_test = {"A": make_split(50), "B": make_split(40)}

    fake_model_builder = lambda: build_model_fn(seq_len, n_features)
    fake_config = build_config({
        "run": {"run_dir": "/tmp/_wiring_selftest_run", "epochs": 1, "batch_size": 32,
                "verbose": 0, "train_mode": "new"},
        "targets": build_target_configs(("high", "low", "close")),
    })
    fake_train_ds = tf.data.Dataset.from_tensor_slices(
        (fake_train[f"X_{tf_name}"], _y_for_targets(fake_train, fake_config))
    ).batch(32, drop_remainder=True)
    fake_val_ds = tf.data.Dataset.from_tensor_slices(
        (fake_val[f"X_{tf_name}"], _y_for_targets(fake_val, fake_config))
    ).batch(32, drop_remainder=True)
    sample = next(iter(fake_train_ds))

    fake_trainer, fake_callbacks, fake_ie = build_training_system(fake_model_builder, fake_config, sample)
    fake_trainer.fit(fake_train_ds, validation_data=fake_val_ds, initial_epoch=fake_ie,
                      epochs=fake_config["run"]["epochs"], callbacks=fake_callbacks, verbose=0)
    fake_model = fake_trainer.model

    fake_test_dict = {}
    for asset, split in fake_test.items():
        last_close = split["last_candles"][:, LAST_CLOSE_COL]
        fake_test_dict[asset] = {
            f"X_{tf_name}": split[f"X_{tf_name}"],
            "base_params": np.stack([last_close, last_close], axis=1).astype("float32"),
            "last_candles": split["last_candles"],
            "y": {t: split["y"][f"y_{t}_reg"] for t in ("high", "low", "close")},
        }

    results = run_full_analysis(
        model=fake_model, test_dict=fake_test_dict, timeframes=[tf_name],
        target_specs=DEFAULT_PRICE_TARGETS, make_plots=False, verbose=False,
        out_dir="/tmp/_wiring_selftest_analysis",
    )
    assert "per_asset_results" in results and len(results["per_asset_results"]) == 2

    if verbose:
        print("✅ نجح اختبار التوصيل: بيانات ← نموذج ← تدريب ← chicks بلا أي استثناء، "
              f"عبر {len(results)} تقريراً/تحليلاً")
    return True


def _y_for_targets(split, config):
    return {cfg["true_key"]: split["y"][cfg["true_key"]] for cfg in config["targets"].values()}


run_wiring_selftest()